# NPTEL Study Notes: Parameter-Efficient Fine-Tuning (PEFT) of LLMs

**Lecture Context:** Fine-Tuning Large Language Models efficiently.

## 1. Evolution of LLM Tuning Paradigms

To understand modern fine-tuning, we must contrast it with earlier paradigms:

* **Transfer Learning (Pre-LLM Era):** Involved two phases. First, **pre-training** on a large unlabeled corpus to learn world knowledge (word/sentence representations). Second, **full fine-tuning** using task-specific labeled data to update *all* model weights for a specific task.
* **LLM Era:** The pre-training phase is now followed by **instruction fine-tuning** and **alignment** (using general-purpose labeled data). The model learns general world knowledge and instruction-following, not just a single specific task.

## 2. In-Context Learning (Prompting) & Its Limitations

In-context learning involves feeding task instructions and input-output examples directly into the LLM's input prompt (also called **hard prompting**).

* **Advantage:** No hosting or training costs for the user. LLMs can be accessed via APIs, making them highly accessible.
* **Disadvantages:**
1. **Suboptimal Performance:** Prompting rarely matches the accuracy of a fully fine-tuned model on critical/complex tasks.
2. **High Sensitivity (Brittleness):** Performance fluctuates wildly based on prompt phrasing, the order of examples, and underlying model updates (e.g., a prompt that works for Llama 3 might fail on Llama 3.1).
3. **Opaque Assumptions:** It is difficult to know exactly what the model is inferring from the prompt versus what it assumes from its pre-trained knowledge.
4. **Inefficiency:** Long prompts (e.g., for document summarization) consume token limits, increase latency, and waste computational throughput.



## 3. The Bottleneck of Full Fine-Tuning

If prompting is insufficient, why not just fully fine-tune the LLM?

* **Hardware Constraints:** Training requires memory not just for the model weights, but for **optimizer states, gradients, and forward activations**. Fully fine-tuning requires memory equivalent to **12 to 20 times** the number of trainable parameters. Fine-tuning a 175B parameter model (like GPT-3) is prohibitively expensive for most organizations.
* **Storage Overhead:** Saving a 350GB checkpoint for *each* specific task is tedious. Hosting 10 different tasks would require hosting 10 separate massive models, wasting energy and hardware.
* **Overfitting & Catastrophic Forgetting:** With limited training data, massive models easily memorize the training set (overfitting) and completely forget their pre-trained world knowledge (catastrophic forgetting).

---

## 4. Parameter-Efficient Fine-Tuning (PEFT)

**Core Concept:** Freeze the majority of the pre-trained LLM's parameters and only train a very small subset of new or existing parameters.

**Advantages of PEFT:**

* **Lower Hardware Requirements:** Fewer trainable parameters mean significantly reduced optimizer states and gradients. Training can be done on older, smaller GPUs (e.g., V100s instead of H100s).
* **Faster Convergence:** Because the world model is frozen, the model quickly learns to map existing knowledge to the new task.
* **Mitigates Overfitting:** Reduced parameter capacity naturally prevents strict memorization of small datasets.
* **Prevents Catastrophic Forgetting:** The frozen base model retains its world knowledge, allowing tasks to generalize better to out-of-domain data.
* **Storage Efficiency:** Only the tiny set of incremental weights needs to be saved and loaded per task.

---

## 5. Key PEFT Techniques

### 5.1 Prompt Tuning (Soft Prompting)

Instead of a human manually engineering text prompts (hard prompting), the model learns continuous, trainable vectors (**soft prompts**).

* **Mechanism:** Special tokens are reserved in the input layer. Only the embeddings of these special task-specific tokens are trained; the rest of the LLM is frozen.
* **Performance:** For smaller models, full fine-tuning wins. However, as model size reaches ~11 Billion parameters (e.g., T5), prompt tuning matches full fine-tuning performance. It also generalizes better to unseen domains (e.g., from Wikipedia data to Books data).
* **Hyperparameters:** Increasing prompt length improves performance up to a point (saturates around 20 tokens for T5).
* **Initialization:** Initializing soft prompts with embeddings of words related to the task yields faster and more stable convergence than random initialization.
* **Operational Advantage (Multitask Serving):** A single LLM can serve 100+ tasks simultaneously in a single batch. The server simply concatenates the specific soft prompt for Task A to User A's input, and Task B's soft prompt to User B's input.

### 5.2 Prefix Tuning

A contemporary to Prompt Tuning, but instead of adding trainable tokens only at the input layer, trainable parameters (**prefixes**) are prepended to the hidden states at **every layer** of the transformer.

* **Mechanism:** In a regular transformer, these prefixes influence every subsequent word in the sequence across all layers.
* **The Unstable Training Hack:** Directly parameterizing the prefixes with an embedding lookup ($f_\theta$) caused unstable training (gradients exploding/vanishing). The authors solved this by reparameterizing using a smaller embedding and a large Multi-Layer Perceptron (MLP), denoted as $f'_\theta$, during training. After training, this large network is collapsed back into a simple embedding matrix to save space.
* **Performance:** Achieves comparable results to full fine-tuning using only **0.1%** of the model's parameters.

### 5.3 Adapters (Houlsby et al.)

Instead of prepending tokens, Adapters insert small, new feed-forward layers *inside* each transformer block.

* **Architecture:** It uses a **bottleneck structure** to reduce parameters.
1. **Down-projection:** Reduces the high-dimensional hidden state (e.g., 1024) to a lower dimension $m$ (e.g., 24).
2. **Non-linearity.**
3. **Up-projection:** Projects back to the original dimension (1024).


* **Residual Connections:** A critical residual connection is added *around* the adapter. The adapter weights are initialized near zero. Therefore, at the start of training, the adapter outputs zero, and the residual connection ensures the transformer behaves exactly like the frozen pre-trained model, ensuring stable training.
* **Performance:** Matches full fine-tuning using only **~3.6%** of parameters.
* **Disadvantage:** **Inference Latency.** Because new layers are physically inserted sequentially into the transformer blocks, it increases computation time and inference latency.

### 5.4 Low-Rank Adaptation (LoRA)

LoRA is currently the most popular PEFT technique, grounded in the theory of **Intrinsic Dimensionality**.

* **Theory (Intrinsic Dimensionality):** Li et al. (2018) showed that models can learn tasks in a much lower-dimensional space $d$ than their total parameter count $D$. Later research (2021) proved that taking the layer-wise *structure* of a transformer into account further reduces the intrinsic dimensionality needed to learn a task.
* **Mechanism:** Instead of updating the massive pre-trained weight matrix $W$, LoRA tracks the weight update $\Delta W$. It factorizes $\Delta W$ into two low-rank matrices, $A$ and $B$.

$$\Delta W = B \times A$$


* If $W$ is $1024 \times 1024$, $\Delta W$ would normally be $1024 \times 1024$.
* With LoRA (rank $r = 8$), matrix $A$ is $8 \times 1024$ and matrix $B$ is $1024 \times 8$. The parameters drop significantly.


* **No Non-linearity:** Unlike Adapters, LoRA has no non-linearity between $B$ and $A$. This is crucial because it allows the matrices to be pre-multiplied and fused directly back into the original weights after training:

$$W_{new} = W + (B \times A)$$


* **Zero Inference Latency:** Because the weights are fused, the architecture remains unchanged. There is zero inference latency overhead.
* **Initialization:** Matrix $A$ is initialized using a Gaussian distribution. Matrix $B$ is initialized to exactly **zero**. Thus, $\Delta W = 0$ at the start of training, preserving base model behavior initially.
* **Application Scope:** Authors recommend applying LoRA to the Attention weights (Query, Key, Value) and the output projection matrix. Ablations show updating just the Query and Value matrices often suffices.
* **Rank ($r$) as Intrinsic Dimensionality:** If a task is highly complex and the base model is weak, a higher Rank ($r$) is needed. If the base model is highly capable and the task is simple, $r=1$ or $r=2$ might suffice.

#### Notable LoRA Extensions:

* **QLoRA:** Quantizes the base model to drastically reduce the memory needed to perform LoRA.
* **Long LoRA:** Optimized for extending the context length of LLMs.
* **LoRA+:** Uses different learning rates for matrices $A$ and $B$ to improve convergence speed.
* **DyLoRA:** Dynamically searches for the optimal rank $r$ during training without needing to run multiple parallel experiments.

---

## 6. Practice Questions

**Q1: Why does prefix tuning require reparameterization with a large MLP ($f'_\theta$) during the training phase?**
**A:** Directly mapping prefix tokens to the high-dimensional hidden space using a simple embedding matrix ($f_\theta$) led to severe training instability. The MLP acts as a stabilizing hack during backpropagation. Once training is complete, the MLP is collapsed back into a standard embedding to save space.

**Q2: Both Adapters and LoRA introduce bottleneck structures to reduce parameters. Why do Adapters increase inference latency while LoRA does not?**
**A:** Adapters insert a new layer containing a non-linearity (activation function) sequentially inside the transformer block. This requires the model to compute an extra sequential step during every forward pass. LoRA factorizes the weight update ($\Delta W = BA$) without any non-linearity. Because it is a linear mathematical update, $BA$ can be fused directly into the pre-trained weights ($W + BA$) after training, resulting in the exact same architectural footprint as the original model.

**Q3: In LoRA, how are matrices A and B initialized, and why is this critical?**
**A:** Matrix A is initialized with a Gaussian distribution (small variance), and Matrix B is initialized to exactly zero. Because $B = 0$, the product $BA = 0$. This ensures that at step 0 of training, the model behaves exactly like the frozen pre-trained LLM, preventing random noise from destroying the model's outputs at the start of fine-tuning.

**Q4: A company wants to serve 150 different specialized NLP tasks using a single base LLM, but traffic for each task fluctuates unpredictably. Which PEFT technique is most structurally suited for this serving infrastructure?**
**A:** **Prompt Tuning (Soft Prompting).** Because only the input embeddings change per task, the server only needs to keep one copy of the heavy base LLM in VRAM. It can dynamically batch queries for different tasks together by simply concatenating the correct soft prompt tokens to each specific input at the tokenizer stage.

---

## 7. Revision Summary

* **In-Context Learning:** API-friendly but brittle, latency-heavy, and suboptimal for critical tasks.
* **Full Fine-Tuning:** Best accuracy, but prohibitive due to memory constraints (optimizer states/gradients) and prone to catastrophic forgetting.
* **PEFT:** Trains ~0.1% to 4% of parameters. Saves memory, storage, and prevents catastrophic forgetting.
* **Prompt Tuning:** Trains virtual tokens at the input layer. Great for scale and multitask batch serving.
* **Prefix Tuning:** Appends trainable tokens to every transformer layer. Needs MLP trick for stability.
* **Adapters:** Inserts bottleneck layers with non-linearity inside transformer blocks. Causes inference latency.
* **LoRA:** Learns a low-rank weight update ($\Delta W = BA$). No inference latency (weights are fused). B is initialized to 0 for stability. Rank ($r$) scales with task complexity.